## 11.14 תרגול מסכם

שלוש שאלות בסגנון מבחן, שמשלבות כמה מהכלים מהשבוע יחד: סטטיסטיקה תיאורית, הולכת שגיאות, התאמה ליניארית (עם/בלי משקלים), וטיב התאמה. לכל שאלה יש תא "עבודה" ריק ופתרון מלא בהמשך -- מומלץ לנסות לבד לפני שפותחים את הפתרון.

In [ ]:
import numpy as np
import pandas as pd

g = 9.8
df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna()

def linear_fit(x, y):
    x_bar, y_bar = x.mean(), y.mean()
    m = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar)**2)
    b = y_bar - m * x_bar
    return m, b

### שאלה 1

עבור זווית 15 מעלות: חשבו את `v0_mean`, `v0_sem`, ואז השתמשו בהולכת שגיאות אנליטית כדי לחשב את `sigma_R` (השגיאה על הטווח התאורטי `R = v0**2/g * sin(2*theta)`). מהו הטווח התאורטי `R` ± `sigma_R`?

In [ ]:
# עבודה עבור שאלה 1


`````{admonition} פתרון
:class: dropdown, tip
```python
angle1 = 15.0
theta1 = np.radians(angle1)
sub1 = df_clean[df_clean["angle_deg"] == angle1]

v0_mean1 = sub1["v0_measured"].mean()
v0_sem1 = sub1["v0_measured"].std(ddof=1) / np.sqrt(len(sub1))

R1 = v0_mean1**2 / g * np.sin(2*theta1)
dR_dv0_1 = 2 * v0_mean1 / g * np.sin(2*theta1)
sigma_R1 = abs(dR_dv0_1) * v0_sem1

print(f"R = {R1:.3f} +/- {sigma_R1:.3f} m")
```
`````

### שאלה 2

עבור זווית 60 מעלות (הזווית עם ה-SEM הגדול ביותר, בגלל הנקודה החריגה שראינו בסעיף 11.1): התאימו קו ישר בין `run_id` ל-`range_measured` **בתוך אותה זווית בלבד**. חשבו את `chi^2_nu` (עם `sigma_y` = סטיית התקן של מדידה בודדת, לא ה-SEM), והחליטו: האם יש עדות לכך שהטווח משתנה שיטתית עם מספר הריצה (`run_id`), או שההתפזרות מוסברת כולה על ידי רעש מדידה?

In [ ]:
# עבודה עבור שאלה 2


`````{admonition} פתרון
:class: dropdown, tip
```python
angle2 = 60.0
sub2 = df_clean[df_clean["angle_deg"] == angle2]
x2 = sub2["run_id"].to_numpy(dtype=float)
y2 = sub2["range_measured"].to_numpy(dtype=float)
sigma2 = y2.std(ddof=1)

m2, b2 = linear_fit(x2, y2)
resid2 = y2 - (m2*x2 + b2)
chi2_2 = np.sum((resid2/sigma2)**2)
reduced_chi2_2 = chi2_2 / (len(x2) - 2)

print(f"שיפוע: {m2:.3f}")
print(f"chi^2_nu: {reduced_chi2_2:.3f}")
```
פרשנות: אם `chi^2_nu` יוצא קרוב ל-1 (או קטן ממנו) והשיפוע קטן יחסית לשגיאתו, אין עדות למגמה אמיתית -- הפיזור מוסבר על ידי רעש מדידה (ולא שכחנו: יש בקבוצה הזו נקודה חריגה, שכבר יודעים שהיא זו שמנפחת את הפיזור, לא מגמה אמיתית).
`````

### שאלה 3

חזרו על ההתאמה הראשית של השבוע (טווח ממוצע לכל זווית מול `sin(2*theta)`, על חמש הזוויות) בשתי גרסאות: **לא-משוקללת** ו-**משוקללת** (לפי `1/sigma_y**2`, כאשר `sigma_y` הוא ה-SEM של הטווח בכל זווית). עבור כל גרסה, חשבו את השיפוע `m` ואת `chi^2_nu`. איזו משתי הגרסאות נותנת שיפוע קרוב יותר לערך התאורטי `v0**2/g` עם `v0=20`?

In [ ]:
# עבודה עבור שאלה 3


`````{admonition} פתרון
:class: dropdown, tip
```python
angles3 = sorted(df_clean["angle_deg"].unique())
x3 = np.array([np.sin(2*np.radians(a)) for a in angles3])
y3 = np.array([df_clean[df_clean["angle_deg"] == a]["range_measured"].mean() for a in angles3])
sigma_y3 = np.array([
    df_clean[df_clean["angle_deg"] == a]["range_measured"].std(ddof=1) / np.sqrt(len(df_clean[df_clean["angle_deg"] == a]))
    for a in angles3
])

def weighted_linear_fit(x, y, sigma_y):
    w = 1.0 / sigma_y**2
    x_bar_w = np.sum(w*x) / np.sum(w)
    y_bar_w = np.sum(w*y) / np.sum(w)
    m_w = np.sum(w * (x - x_bar_w) * (y - y_bar_w)) / np.sum(w * (x - x_bar_w)**2)
    b_w = y_bar_w - m_w * x_bar_w
    return m_w, b_w

def reduced_chi2(x, y, sigma_y, m, b):
    resid = y - (m*x + b)
    chi2 = np.sum((resid/sigma_y)**2)
    return chi2 / (len(x) - 2)

m_uw, b_uw = linear_fit(x3, y3)
m_w, b_w = weighted_linear_fit(x3, y3, sigma_y3)

theoretical = 20.0**2 / g

print(f"לא-משוקלל: m={m_uw:.3f}  (הפרש מהתאורטי: {abs(m_uw-theoretical):.3f})   chi^2_nu={reduced_chi2(x3,y3,sigma_y3,m_uw,b_uw):.3f}")
print(f"משוקלל:    m={m_w:.3f}  (הפרש מהתאורטי: {abs(m_w-theoretical):.3f})   chi^2_nu={reduced_chi2(x3,y3,sigma_y3,m_w,b_w):.3f}")
print(f"תאורטי:    v0^2/g = {theoretical:.3f}")
```
פרשנות: ההתאמה המשוקללת נותנת פחות משקל לזווית 60 (שם ה-SEM גדול, בגלל הנקודה החריגה) -- הרעיון התאורטי הוא שנקודות פחות מהימנות ישפיעו פחות. בפועל, עם דגימה קטנה (5 נקודות בלבד) ורעש אקראי, זה לא מבטיח ששיפוע ה-`m` המשוקלל יהיה תמיד המדויק ביותר בהשוואה ל-`v0**2/g` התאורטי בכל הרצה -- אבל `chi^2_nu` המשוקלל **כן** צפוי, באופן עקבי יותר, להיות קרוב יותר ל-1: הוא מודד עד כמה ההתאמה עצמה מתיישבת עם שגיאות המדידה המוצהרות, לא עד כמה `m` קרוב לערך התאורטי (שאותו לא תמיד יודעים מראש במעבדה אמיתית).
`````